In [1]:
# Python 3.12
# pip install yfinance pandas

import os
import yfinance as yf
import pandas as pd

MAG7 = ["AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "TSLA"]

SAVE_DIR = r"C:\Users\Admin\Desktop\Project\data\fundamentals"
os.makedirs(SAVE_DIR, exist_ok=True)


def tidy(df, ticker, frequency, statement):
    """Return tidy long fundamentals."""
    if df is None or df.empty:
        return None

    out = (
        df.T
        .reset_index()
        .rename(columns={"index": "period_end_date"})
        .melt(id_vars=["period_end_date"], var_name="line_item", value_name="value")
    )

    out["ticker"] = ticker
    out["frequency"] = frequency
    out["statement"] = statement
    out["period_end_date"] = pd.to_datetime(out["period_end_date"])

    out = out.dropna(subset=["value"])

    return out[["ticker", "period_end_date", "frequency", "statement", "line_item", "value"]]


annual, quarterly = [], []

for symbol in MAG7:
    t = yf.Ticker(symbol)

    annual += [
        tidy(t.financials, symbol, "annual", "income_statement"),
        tidy(t.balance_sheet, symbol, "annual", "balance_sheet"),
        tidy(t.cashflow, symbol, "annual", "cashflow"),
    ]

    quarterly += [
        tidy(t.quarterly_financials, symbol, "quarterly", "income_statement"),
        tidy(t.quarterly_balance_sheet, symbol, "quarterly", "balance_sheet"),
        tidy(t.quarterly_cashflow, symbol, "quarterly", "cashflow"),
    ]


annual_df = pd.concat([x for x in annual if x is not None], ignore_index=True)
quarterly_df = pd.concat([x for x in quarterly if x is not None], ignore_index=True)


annual_df.to_csv(os.path.join(SAVE_DIR, "mag7_fundamentals_annual.csv"), index=False)
quarterly_df.to_csv(os.path.join(SAVE_DIR, "mag7_fundamentals_quarterly.csv"), index=False)

print("Done.")


Done.
